# Text Features — Raw Daily Text Aggregation

Aggregates raw text per day from three sources:
- **Bluesky** posts
- **Reddit** post titles (r/politics, r/Conservative, r/worldnews)
- **MediaCloud News** headlines

Output: `Data/3_Gold/text_raw_daily.csv` — one row per day, one text column per source.

## Why save raw text instead of pre-computed TF-IDF?

TF-IDF must be fitted **only on the training data of each CV fold**.

If we pre-compute TF-IDF on the full train+val region, the IDF weights are computed
using data from *later folds* when evaluating *earlier folds* — a form of data leakage.

Correct fit/transform schema per fold:

```
Fold 1:  FIT TF-IDF on 35 train days  →  TRANSFORM 35 train + 26 val days
Fold 2:  FIT TF-IDF on 52 train days  →  TRANSFORM 52 train + 26 val days
Fold 3:  FIT TF-IDF on 78 train days  →  TRANSFORM 78 train + 26 val days
Test:    FIT TF-IDF on 110 TV days    →  TRANSFORM 110 TV   + 14 test days
```

The TF-IDF fitting happens inside the CV loop in `models.ipynb` using a
`sklearn.compose.ColumnTransformer` — this guarantees correct fit/transform
per fold automatically.

<!-- toc -->
## Contents
- [Why save raw text instead of pre-computed TF-IDF?](#why-save-raw-text-instead-of-pre-computed-tf-idf)
- [Setup](#setup)
- [Load & align on basetable dates](#load-align-on-basetable-dates)
- [Quick vocabulary check](#quick-vocabulary-check)
- [Save](#save)


## Setup

In [1]:
import sys, os, json as _json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from Descriptive.house_style import (apply_style, styled_fig, style_ax,
    BG_DARK, BG_PANEL, TEXT_PRIMARY, TEXT_MUTED, GRID_COLOR, SPINE_COLOR, PALETTE)
apply_style()

## Load & align on basetable dates

In [2]:
basetable = pd.read_csv('../Data/3_Gold/basetable.csv', parse_dates=['date'])
basetable = basetable.sort_values('date').reset_index(drop=True)
DATE_MIN  = basetable['date'].min()
DATE_MAX  = basetable['date'].max()

# ── Bluesky ───────────────────────────────────────────────────────────────────
bsky_raw = pd.read_csv('../Data/2_Silver/Bluesky/bsky_US_2024_posts.csv',
                        usecols=['timestamp', 'text'])
bsky_raw['date'] = (pd.to_datetime(bsky_raw['timestamp'], format='ISO8601', utc=True)
                      .dt.normalize().dt.tz_localize(None))
bsky_raw = bsky_raw[(bsky_raw['date'] >= DATE_MIN) & (bsky_raw['date'] <= DATE_MAX)]
bsky_daily = (bsky_raw.groupby('date')['text']
              .apply(lambda x: ' '.join(x.dropna().astype(str)))
              .rename('bsky_text').reset_index())

# ── Reddit titles ─────────────────────────────────────────────────────────────
reddit_rows = []
for fname in ['r_politics_posts.jsonl', 'r_conservative_posts.jsonl',
              'r_worldnews_posts.jsonl']:
    with open(f'../Data/1_Bronze/Reddit/{fname}', encoding='utf-8') as fh:
        for line in fh:
            d = _json.loads(line)
            reddit_rows.append({'title': d.get('title', ''),
                                 'ts': d.get('created_utc') or d.get('created')})
reddit_df = pd.DataFrame(reddit_rows)
reddit_df['date'] = (pd.to_datetime(reddit_df['ts'], unit='s', utc=True)
                       .dt.normalize().dt.tz_localize(None))
reddit_df = reddit_df[(reddit_df['date'] >= DATE_MIN) & (reddit_df['date'] <= DATE_MAX)]
reddit_daily = (reddit_df.groupby('date')['title']
                .apply(lambda x: ' '.join(x.dropna().astype(str)))
                .rename('reddit_text').reset_index())

# ── News headlines ────────────────────────────────────────────────────────────
mc = pd.read_csv('../Data/1_Bronze/Newspapers/mediacloud_stories.csv',
                  usecols=['date', 'title', 'url'], parse_dates=['date'])
mc = mc[(mc['date'] >= DATE_MIN) & (mc['date'] <= DATE_MAX)]
news_daily = (mc.drop_duplicates('url')
              .groupby('date')['title']
              .apply(lambda x: ' '.join(x.dropna().astype(str)))
              .rename('news_text').reset_index())

# ── Merge on master date index — fill missing days with empty string ──────────
text_df = (basetable[['date']]
           .merge(bsky_daily,   on='date', how='left')
           .merge(reddit_daily, on='date', how='left')
           .merge(news_daily,   on='date', how='left'))

for col in ['bsky_text', 'reddit_text', 'news_text']:
    text_df[col] = text_df[col].fillna('')

print(f"Shape: {text_df.shape}")
for col in ['bsky_text', 'reddit_text', 'news_text']:
    avg_words = text_df[col].apply(lambda x: len(x.split())).mean()
    empty     = (text_df[col] == '').sum()
    print(f"  {col:<15}  avg {avg_words:6.0f} words/day  |  {empty} empty days")

Shape: (124, 4)
  bsky_text        avg   5702 words/day  |  1 empty days


  reddit_text      avg  14036 words/day  |  2 empty days


  news_text        avg  14323 words/day  |  6 empty days


## Quick vocabulary check

In [3]:
# Show most common non-stopword terms per source to verify text quality
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def top_terms(texts, n=15):
    cnt = Counter()
    for t in texts:
        for w in t.lower().split():
            if w.isalpha() and w not in ENGLISH_STOP_WORDS and len(w) > 2:
                cnt[w] += 1
    return cnt.most_common(n)

for col, label in [('bsky_text','Bluesky'), ('reddit_text','Reddit'), ('news_text','News')]:
    terms = top_terms(text_df[col])
    print(f"{label}: {', '.join(t for t,_ in terms)}")

Bluesky: trump, house, die, just, like, harris, latest, und, der, donald, people, vote, kamala, introduced, committee


Reddit: trump, harris, kamala, biden, says, election, new, donald, campaign, president, vance, democrats, israel, rally, presidential


News: trump, harris, kamala, biden, election, says, donald, campaign, new, debate, presidential, vance, rally, assassination, news


## Save

In [4]:
OUT = '../Data/3_Gold/text_raw_daily.csv'
text_df.to_csv(OUT, index=False)
print(f"Saved {OUT}  ({len(text_df)} rows x {len(text_df.columns)} cols)")
print("Columns:", list(text_df.columns))
print()
print("Next step: run models.ipynb — TF-IDF is fitted per CV fold inside the pipeline.")

Saved ../Data/3_Gold/text_raw_daily.csv  (124 rows x 4 cols)
Columns: ['date', 'bsky_text', 'reddit_text', 'news_text']

Next step: run models.ipynb — TF-IDF is fitted per CV fold inside the pipeline.
